# LSTM — Long Short-Term Memory (Turkey, Pipeline B)

**Model**: `nowcast_lstm.LSTM`  
**Variables**: Cat3 + COVID (25 vars)  
**Train**: 1995-Q2 to 2011-Q4 / **Val**: 2012-Q1 to 2017-Q4 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: Internal to library (StandardScaler equivalent)  
**HP tuning**: YES (n_hidden, n_layers, dropout tuned on validation window)  
**n_timesteps**: 6 (months)

In [1]:
import sys, os
import numpy as np
import pandas as pd
from nowcast_lstm.LSTM import LSTM
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams['figure.figsize'] = [15, 10]

sys.path.insert(0, os.path.join(os.path.pardir, 'turkey_data'))
from turkey_helpers import (
    gen_lagged_data, get_features, load_data,
    PREDICTIONS_DIR, TARGET, COVID
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_START = '1995-01-01'
TRAIN_END   = '2011-12-31'
VAL_END     = '2017-12-31'
TEST_START  = '2018-01-01'
TEST_END    = '2025-12-31'
N_TIMESTEPS = 6
VINTAGES    = {'m1': -2, 'm2': -1, 'm3': 0}

print('LSTM (Turkey) — Cat3, n_timesteps={}'.format(N_TIMESTEPS))

LSTM (Turkey) — Cat3, n_timesteps=6


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
print('Features: {}'.format(len(features)))

# Phase A: HP tuning on validation window (2012-2017)
# Create a single validation-period dataset
val_data = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= VAL_END)].reset_index(drop=True)
# BUG FIX: Target variable explicitly included
val_data = val_data[['date', TARGET] + features]  # Cat3 + Target vars only
# Use m3 vintage (most data available) for tuning
val_masked = gen_lagged_data(metadata, val_data.copy(), pd.Timestamp(VAL_END), lag=0)

print('Tuning HPs on validation window...')
best_score = np.inf
best_hps = {'n_hidden': 32, 'n_layers': 1, 'dropout': 0.0}

# Light grid — 8 combinations
for n_hidden in [32, 64]:
    for n_layers in [1, 2]:
        for dropout in [0.0, 0.2]:
            try:
                model = LSTM(
                    data=val_masked, target_variable=TARGET,
                    n_timesteps=N_TIMESTEPS,
                    fill_na_func=np.nanmean, fill_ragged_edges_func=np.nanmean,
                    n_models=5, train_episodes=50, batch_size=50,
                    decay=0.98, n_hidden=n_hidden, n_layers=n_layers,
                    dropout=dropout,
                    criterion=torch.nn.MSELoss(),
                    optimizer=torch.optim.Adam,
                    optimizer_parameters={'lr': 1e-2, 'weight_decay': 0.0},
                )
                model.train(quiet=True)
                preds = model.predict(val_masked)
                # Evaluate on Q-end months only
                q_preds = preds[preds.date.dt.month.isin([3, 6, 9, 12])]
                actual_mask = ~pd.isna(q_preds[TARGET])
                if actual_mask.sum() > 5:
                    score = np.nanmean((q_preds.loc[actual_mask, 'predictions'].values -
                                         q_preds.loc[actual_mask, TARGET].values) ** 2)
                    if score < best_score:
                        best_score = score
                        best_hps = {'n_hidden': n_hidden, 'n_layers': n_layers, 'dropout': dropout}
            except Exception:
                pass

print('Best HPs: n_hidden={}, n_layers={}, dropout={}'.format(
    best_hps['n_hidden'], best_hps['n_layers'], best_hps['dropout']))

Features: 25


Tuning HPs on validation window...


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Best HPs: n_hidden=32, n_layers=1, dropout=0.0


In [3]:
# Phase B: Rolling test loop with frozen HPs
test_quarters = monthly[(monthly['date'] >= TEST_START) &
                         (monthly['date'] <= TEST_END) &
                         monthly['date'].dt.month.isin([3, 6, 9, 12])]['date'].tolist()

predictions = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actual = monthly[monthly['date'] == pd_q][TARGET].values[0]
    actuals_list.append(actual)

    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= fc)].reset_index(drop=True)
        # BUG FIX: Target variable explicitly included
        train = train[['date', TARGET] + features]  # Cat3 + Target vars only
        train_m = gen_lagged_data(metadata, train.copy(), fc, lag=0)

        try:
            model = LSTM(
                data=train_m, target_variable=TARGET,
                n_timesteps=N_TIMESTEPS,
                fill_na_func=np.nanmean, fill_ragged_edges_func=np.nanmean,
                n_models=10, train_episodes=100, batch_size=50,
                decay=0.98,
                n_hidden=best_hps['n_hidden'],
                n_layers=best_hps['n_layers'],
                dropout=best_hps['dropout'],
                criterion=torch.nn.MSELoss(),
                optimizer=torch.optim.Adam,
                optimizer_parameters={'lr': 1e-2, 'weight_decay': 0.0},
            )
            model.train(quiet=True)

            # Predict on the same data (library handles date filtering)
            pred_df = model.predict(train_m)
            pred_row = pred_df[pred_df['date'] == fc]
            if len(pred_row) > 0:
                pred = pred_row['predictions'].values[0]
            else:
                pred = pred_df['predictions'].values[-1]  # last prediction
        except Exception:
            pred = np.nan
            failed += 1

        predictions[vn].append(pred)

    if (i + 1) % 4 == 0:
        print('  {} / {}'.format(i + 1, len(test_quarters)))

print('Done. {} quarters, {} failures.'.format(len(test_quarters), failed))

  4 / 32


  8 / 32


  12 / 32


  16 / 32


  20 / 32


  24 / 32


  28 / 32


  32 / 32
Done. 32 quarters, 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    df = pd.DataFrame({
        'date': test_quarters, 'actual': actuals_list,
        'prediction': predictions[vn],
    })
    path = os.path.join(PREDICTIONS_DIR, 'lstm_{}.csv'.format(vn))
    df.to_csv(path, index=False)
    print('Saved {} ({} rows) pred=[{:.4f},{:.4f}]'.format(
        os.path.basename(path), len(df), df['prediction'].min(), df['prediction'].max()))

Saved lstm_m1.csv (32 rows) pred=[-0.0444,0.0218]
Saved lstm_m2.csv (32 rows) pred=[0.0040,0.0576]
Saved lstm_m3.csv (32 rows) pred=[-0.0203,0.0452]


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m] - p[m]) ** 2)) if m.sum() > 0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m] - p[m])) if m.sum() > 0 else np.nan

panels = {
    'Pre-crisis (2018-2019)': ('2018-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2018-2025)': ('2018-01-01', '2025-12-31'),
}
a = np.array(actuals_list)
d = pd.to_datetime(test_quarters)
print('LSTM (Turkey) — Evaluation by Panel & Vintage')
print('LSTM HPs: n_hidden={}, n_layers={}, dropout={}'.format(
    best_hps['n_hidden'], best_hps['n_layers'], best_hps['dropout']))
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print('--- {} ---'.format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print('  {}  RMSFE={:.6f}  MAE={:.6f}  N={}'.format(
            vn, rmse(a[m], pp[m]), mae(a[m], pp[m]), m.sum()))

LSTM (Turkey) — Evaluation by Panel & Vintage
LSTM HPs: n_hidden=32, n_layers=1, dropout=0.0
--- Pre-crisis (2018-2019) ---
  m1  RMSFE=0.018190  MAE=0.013331  N=8
  m2  RMSFE=0.018797  MAE=0.013915  N=8
  m3  RMSFE=0.018082  MAE=0.013557  N=8
--- COVID      (2020-2021) ---
  m1  RMSFE=0.085984  MAE=0.053838  N=7
  m2  RMSFE=0.073540  MAE=0.051085  N=7
  m3  RMSFE=0.052314  MAE=0.035874  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.011987  MAE=0.008385  N=16
  m2  RMSFE=0.011872  MAE=0.008125  N=16
  m3  RMSFE=0.011990  MAE=0.008170  N=16
--- Full test  (2018-2025) ---
  m1  RMSFE=0.042119  MAE=0.019563  N=32
  m2  RMSFE=0.036656  MAE=0.018956  N=32
  m3  RMSFE=0.027467  MAE=0.015581  N=32
